# Big Data Analytics: Sentiment Analysis of Amazon Electronics Reviews

**Course:** CAP 4786 – Topics in Big Data  
**Project:** Sentiment Analysis of Customer Reviews  
**Dataset:** Amazon Reviews 2023 (Electronics Category)

This notebook contains the complete PySpark pipeline for loading, preprocessing, modeling, and evaluating sentiment on a large-scale dataset of Amazon customer reviews. It is designed to be run on a Google Cloud Platform (GCP) Dataproc cluster.

## 1. Setup and Initialization

First, we need to install the Hugging Face `datasets` library to stream the data directly into our cluster without manual downloading.

In [ ]:
!pip install datasets

Initialize the PySpark session and import necessary libraries.

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, when, length, lower, regexp_replace
from pyspark.sql.types import StringType, FloatType, IntegerType
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from datasets import load_dataset
import pandas as pd
import time

# Create SparkSession (already created as 'spark' in Dataproc Jupyter by default, but good practice to include)
spark = SparkSession.builder \
    .appName("Amazon_Electronics_Sentiment_Analysis") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

## 2. Data Acquisition (Streaming from Hugging Face)

We will load a subset of the "Electronics" category reviews. To demonstrate scalability while managing resource constraints, we will convert a Pandas DataFrame slice into a Spark DataFrame.

In [ ]:
print("Loading dataset from Hugging Face...")
start_time = time.time()

# Load the dataset in streaming mode (avoids downloading the entire 43.9M rows at once to memory)
dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", "raw_review_Electronics", split="full", streaming=True, trust_remote_code=True)

# For this project, we will extract a representative subset (e.g., 500,000 reviews for testing/development)
# In your final run, you can increase this number (e.g., to 5,000,000) to test scalability.
subset_size = 500000

# Extract data into a list of dictionaries
data_list = []
for i, row in enumerate(dataset):
    if i >= subset_size:
        break
    # We only need the rating and the text for sentiment analysis
    if row['text'] is not None and row['rating'] is not None:
        data_list.append({'rating': float(row['rating']), 'text': str(row['text'])})

# Convert to Pandas DataFrame first, then to Spark DataFrame
pdf = pd.DataFrame(data_list)
df = spark.createDataFrame(pdf)

end_time = time.time()
print(f"Loaded {df.count()} rows in {end_time - start_time:.2f} seconds.")
df.show(5, truncate=True)

## 3. Data Preprocessing & Feature Engineering

We need to clean the text and create our target variable (Sentiment Label) based on the star rating.

**Baseline Mapping:**
*   Ratings 4.0 - 5.0 -> Positive (Label 1)
*   Ratings 1.0 - 2.0 -> Negative (Label 0)
*   Ratings 3.0 -> Neutral (We will filter these out for a clearer binary classification task)

In [ ]:
# 1. Filter out neutral reviews (rating == 3.0)
df_filtered = df.filter(col("rating") != 3.0)

# 2. Create Binary Sentiment Label
df_labeled = df_filtered.withColumn("label", when(col("rating") >= 4.0, 1).otherwise(0))

# 3. Clean Text (lowercase, remove punctuation)
df_clean = df_labeled.withColumn("clean_text", lower(col("text")))
df_clean = df_clean.withColumn("clean_text", regexp_replace(col("clean_text"), "[^a-zA-Z\\s]", ""))

# Show class distribution (Checking for data skew)
print("Class Distribution:")
df_clean.groupBy("label").count().show()

## 4. Spark ML Pipeline (TF-IDF + Logistic Regression)

We will build a pipeline that:
1.  **Tokenizes** the text into words.
2.  **Removes stop words** (like "the", "is", "and").
3.  Calculates **Term Frequency (TF)** using HashingTF.
4.  Calculates **Inverse Document Frequency (IDF)**.
5.  Trains a **Logistic Regression** model.

In [ ]:
# Define pipeline stages
tokenizer = Tokenizer(inputCol="clean_text", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
hashingTF = HashingTF(inputCol="filtered_words", outputCol="rawFeatures", numFeatures=10000)
idf = IDF(inputCol="rawFeatures", outputCol="features")
lr = LogisticRegression(maxIter=10, regParam=0.01, featuresCol="features", labelCol="label")

# Create the pipeline
pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, idf, lr])

# Split data into Training (80%) and Testing (20%) sets
train_data, test_data = df_clean.randomSplit([0.8, 0.2], seed=42)
print(f"Training Data Count: {train_data.count()}")
print(f"Testing Data Count: {test_data.count()}")

## 5. Model Training

This is where the distributed computing power of Spark shines.

In [ ]:
print("Training the model (this may take a few minutes depending on cluster size)...")
train_start = time.time()

# Fit the pipeline to the training data
model = pipeline.fit(train_data)

train_end = time.time()
print(f"Model training completed in {train_end - train_start:.2f} seconds.")

## 6. Evaluation

We will evaluate the model's performance on the unseen test dataset using standard classification metrics: Accuracy and F1-Score.

In [ ]:
# Make predictions on the test data
predictions = model.transform(test_data)

# Select example rows to display
predictions.select("clean_text", "label", "prediction", "probability").show(5, truncate=50)

# Evaluate the model
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator_acc.evaluate(predictions)

evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
f1_score = evaluator_f1.evaluate(predictions)

print(f"Test Accuracy: {accuracy * 100:.2f}%")
print(f"Test F1-Score: {f1_score * 100:.2f}%")

## 7. Scalability Test (Optional for Presentation)

To satisfy the "Big Data" requirement of the course, you should record the `Model training completed in X seconds` time from Step 5. 

**Action for your report:**
1. Run this notebook with `subset_size = 500,000` and record the training time.
2. Change `subset_size = 2,000,000` (in Step 2), restart the kernel, run again, and record the training time.
3. Change `subset_size = 5,000,000`, run again, and record the time.
4. Plot these three data points in a line chart for your final presentation to demonstrate how Spark handles increasing data loads.

## 8. Real-Time Sentiment Prediction (Interactive Demo)

This section allows you to type in any review text and instantly see the model's sentiment prediction and confidence score. This is ideal for demonstrating the model live during your final presentation.

Simply change the text inside the `user_review` variable below and run the cell.

In [ ]:
import re

def predict_sentiment(review_text):
    """
    Takes a raw review string, runs it through the trained Spark ML pipeline,
    and prints the predicted sentiment with confidence score.
    """
    # --- Preprocess the input the same way as training data ---
    clean = review_text.lower()
    clean = re.sub(r'[^a-zA-Z\s]', '', clean)

    # Create a single-row Spark DataFrame
    input_df = spark.createDataFrame([(clean,)], ["clean_text"])

    # Run through the trained pipeline (skips the LR training step — uses saved model)
    result = model.transform(input_df)

    # Extract prediction and probability
    row = result.select("prediction", "probability").collect()[0]
    prediction = int(row["prediction"])
    prob = row["probability"]
    confidence = max(prob[0], prob[1]) * 100

    sentiment = "POSITIVE" if prediction == 1 else "NEGATIVE"
    emoji = "✅" if prediction == 1 else "❌"

    print("=" * 60)
    print(f"  Review : {review_text}")
    print("-" * 60)
    print(f"  Sentiment  : {emoji}  {sentiment}")
    print(f"  Confidence : {confidence:.1f}%")
    print("=" * 60)


# ---------------------------------------------------------------
# CHANGE THIS TEXT TO ANY REVIEW YOU WANT TO TEST
# ---------------------------------------------------------------
user_review = "This hot sauce is absolutely amazing, best I have ever tasted!"
# ---------------------------------------------------------------

predict_sentiment(user_review)

In [ ]:
# --- Batch Demo: Test multiple reviews at once ---
# Great for showing a variety of predictions during your presentation

sample_reviews = [
    "Absolutely love this product, it arrived quickly and works perfectly!",
    "Terrible quality. Fell apart after two days. Do not buy.",
    "It is okay, nothing special but does the job.",
    "Best coffee I have ever had. Will definitely order again.",
    "The packaging was damaged and the item was completely stale."
]

print("BATCH SENTIMENT PREDICTIONS")
print("=" * 60)
for review in sample_reviews:
    predict_sentiment(review)
    print()